# THIRU Manuscript: Analysis and Figure Generation
## Benchmarking Deep Learning Methods for Retinal EM Synapse Segmentation
Generates publication-quality figures for Scientific Reports submission.
- **Figure 1**: Qualitative comparison of all 4 structures (mito, membrane, ribbon, vesicle)
- **Figure 2**: Method comparison bar chart (all 4 structures, best Dice per method)
- **Figure 3**: Threshold sweep curves (all 4 structures)
- **Figure 4**: Final optimized performance summary (all 4 structures)
- **Figure 5**: 3D reconstruction of ground truth annotations (marching cubes meshes)
- **Figure 6**: Qualitative full-montage evaluation (fine-tuned 2D vs pretrained 3D zero-shot)
- **Part C**: Post-hoc 3D consistency evaluation (3D-CC, z-smoothing, instance tracking)

### Compute Environment
- Figures 1–4: Local (uses pre-computed CSV data + saved masks)
- Figure 5: Requires node01 GPU server (GT labels on `/mnt/Projects/`)
- Figure 6 + Part C: Requires node01 GPU server (full 7000×7000 montages + model checkpoints)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import tifffile
from pathlib import Path
from PIL import Image

# Paths
DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Publication settings
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

# Colors (consistent across all figures)
MITO_COLOR = (0/255, 200/255, 0/255)       # green
MEMBRANE_COLOR = (0/255, 100/255, 255/255)  # blue
RIBBON_COLOR = (200/255, 0/255, 200/255)    # magenta
VESICLE_COLOR = (230/255, 150/255, 0/255)   # orange

# Load data
all_runs = pd.read_csv(DATA_DIR / "all_runs_comparison.csv")
final_best = pd.read_csv(DATA_DIR / "final_best_results.csv")
combo_results = pd.read_csv(DATA_DIR / "combination_optimization" / "all_combination_results.csv")
method_comparison = pd.read_csv(DATA_DIR / "combination_optimization" / "method_comparison.csv")

print("Data loaded successfully")
print(f"All runs: {len(all_runs)} rows")
print(f"Combination results: {len(combo_results)} rows")
print(f"Final best results:")
print(final_best[["structure","best_dice","source_run","threshold_used","tta_applied","ensemble_applied"]].to_string(index=False))

Data loaded successfully
All runs: 20 rows
Combination results: 324 rows
Final best results:
   structure  best_dice     source_run  threshold_used  tta_applied  ensemble_applied
mitochondria   0.869300   ensemble_max            0.35         True              True
    membrane   0.892483       finetune            0.40        False             False
      ribbon   0.762000       finetune            0.80        False             False
    vesicles   0.205900 retrain_ribbon            0.05        False             False


In [2]:
# Figure 1: Qualitative comparison -- EM image with GT vs predicted overlays
# All 4 structures: mitochondria, membrane, ribbon, vesicle
# Uses the saved post-processed prediction masks (same masks used for metrics)

fig, axes = plt.subplots(4, 3, figsize=(7.5, 10.4))

# Overlay colors: lighter shade for GT, darker/brighter shade for prediction
GT_COLORS = {
    "mito": (0.2, 0.9, 0.2),        # light green
    "membrane": (0.3, 0.55, 1.0),    # light blue
    "ribbon": (0.9, 0.2, 0.9),       # magenta
    "vesicle": (1.0, 0.65, 0.0),     # orange
}
PRED_COLORS = {
    "mito": (0.0, 0.55, 0.0),       # dark green
    "membrane": (0.0, 0.2, 0.7),     # dark blue
    "ribbon": (0.6, 0.0, 0.6),       # dark magenta
    "vesicle": (0.8, 0.3, 0.0),      # dark orange
}

structures = [
    ("mito", "Mitochondria"),
    ("membrane", "Membrane"),
    ("ribbon", "Ribbon"),
    ("vesicle", "Vesicle"),
]

for row_idx, (struct_key, struct_name) in enumerate(structures):
    # Load raw image
    raw = tifffile.imread(DATA_DIR / "images" / f"{struct_key}_image_section_002.tif")
    # Load GT label
    gt = tifffile.imread(DATA_DIR / "images" / f"{struct_key}_label_section_002.tif")
    # Load saved post-processed prediction mask
    pred = tifffile.imread(DATA_DIR / "images" / f"{struct_key}_pred_mask_best.tif")
    
    # Normalize raw to 0-1 for display
    raw_norm = raw.astype(np.float32)
    if raw_norm.max() > 0:
        raw_norm = (raw_norm - raw_norm.min()) / (raw_norm.max() - raw_norm.min())
    
    gt_mask = (gt > 0).astype(np.float32)
    pred_mask = (pred > 0).astype(np.float32)
    
    gt_color = GT_COLORS[struct_key]
    pred_color = PRED_COLORS[struct_key]
    
    # Panel A: Raw EM image
    axes[row_idx, 0].imshow(raw_norm, cmap="gray", vmin=0, vmax=1)
    axes[row_idx, 0].set_title("EM Image" if row_idx == 0 else "")
    axes[row_idx, 0].set_ylabel(struct_name, fontsize=11, fontweight="bold")
    axes[row_idx, 0].set_xticks([])
    axes[row_idx, 0].set_yticks([])
    
    # Panel B: GT overlay (lighter shade)
    rgb_gt = np.stack([raw_norm]*3, axis=-1)
    for c in range(3):
        rgb_gt[:,:,c] = np.where(gt_mask > 0,
            rgb_gt[:,:,c] * 0.45 + gt_color[c] * 0.55,
            rgb_gt[:,:,c])
    axes[row_idx, 1].imshow(rgb_gt)
    axes[row_idx, 1].set_title("Ground Truth" if row_idx == 0 else "")
    axes[row_idx, 1].set_xticks([])
    axes[row_idx, 1].set_yticks([])
    
    # Panel C: Prediction overlay (darker shade)
    rgb_pred = np.stack([raw_norm]*3, axis=-1)
    for c in range(3):
        rgb_pred[:,:,c] = np.where(pred_mask > 0,
            rgb_pred[:,:,c] * 0.45 + pred_color[c] * 0.55,
            rgb_pred[:,:,c])
    axes[row_idx, 2].imshow(rgb_pred)
    axes[row_idx, 2].set_title("Prediction" if row_idx == 0 else "")
    axes[row_idx, 2].set_xticks([])
    axes[row_idx, 2].set_yticks([])

plt.tight_layout(w_pad=0.5, h_pad=0.8)

fig.savefig(FIG_DIR / "fig1_qualitative.png", dpi=300)
fig.savefig(FIG_DIR / "fig1_qualitative.tiff", dpi=300)
plt.close(fig)
print("Figure 1 saved: qualitative comparison (4 structures)")

Figure 1 saved: qualitative comparison (4 structures)


In [3]:
# Figure 2: Method comparison bar chart — all 4 structures
# Shows 6 benchmarked methods for mito+membrane, plus ribbon and vesicle best methods
# Grouped by structure for clear comparison

method_labels = [
    "Unadapted",
    "Domain\nAdapted",
    "Fine-tuned\n(Dice)",
    "Fine-tuned\n(Focal)",
    "DA + FT",
    "micro-SAM\nZero-shot",
    "Ribbon\nFocal FT",
    "Vesicle\nRibbon-prox",
]

# Dice scores per structure per method (N/A = 0 for methods not tested on that structure)
mito_scores =     [0.000, 0.000, 0.155, 0.836, 0.844, 0.250, 0, 0]
membrane_scores = [0.000, 0.000, 0.361, 0.870, 0.858, 0.293, 0, 0]
ribbon_scores =   [0, 0, 0, 0, 0, 0, 0.762, 0]
vesicle_scores =  [0, 0, 0, 0, 0, 0, 0, 0.206]

x = np.arange(len(method_labels))
width = 0.20

fig, ax = plt.subplots(figsize=(9, 4.5))

bars1 = ax.bar(x - 1.5*width, mito_scores, width, label="Mitochondria",
               color=MITO_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)
bars2 = ax.bar(x - 0.5*width, membrane_scores, width, label="Membrane",
               color=MEMBRANE_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)
bars3 = ax.bar(x + 0.5*width, ribbon_scores, width, label="Ribbon",
               color=RIBBON_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)
bars4 = ax.bar(x + 1.5*width, vesicle_scores, width, label="Vesicles",
               color=VESICLE_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)

# Add value labels on non-zero bars
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        if height > 0.02:
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.012,
                    f"{height:.3f}", ha="center", va="bottom", fontsize=6.5)

# Add vertical separator before ribbon/vesicle methods
ax.axvline(x=5.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(3, 1.0, "6-method benchmark\n(mito + membrane)", ha="center", fontsize=7,
        fontstyle="italic", color="gray")
ax.text(6.5, 1.0, "Extended\nstructures", ha="center", fontsize=7,
        fontstyle="italic", color="gray")

ax.set_ylabel("Dice Coefficient")
ax.set_xticks(x)
ax.set_xticklabels(method_labels, fontsize=7.5)
ax.set_ylim(0, 1.12)
ax.legend(loc="upper left", framealpha=0.9, ncol=2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axhline(y=0, color="black", linewidth=0.5)

plt.tight_layout()
fig.savefig(FIG_DIR / "fig2_method_comparison.png", dpi=300)
fig.savefig(FIG_DIR / "fig2_method_comparison.tiff", dpi=300)
plt.close(fig)
print("Figure 2 saved: method comparison (all 4 structures)")

Figure 2 saved: method comparison (all 4 structures)


In [4]:
# Figure 3: Threshold sweep curves — all 4 structures
# Mito + membrane from combo_results CSV; ribbon + vesicle from hardcoded eval data

fig, axes = plt.subplots(2, 2, figsize=(7.5, 6))

# --- Mitochondria (top-left) ---
ax = axes[0, 0]
thresh_data = combo_results[combo_results["phase"] == "A_threshold"]
method_styles = {
    "finetune": {"label": "Fine-tuned (focal)", "color": "#2196F3", "ls": "-", "marker": "o"},
    "da_then_finetune": {"label": "DA + FT", "color": "#FF9800", "ls": "--", "marker": "s"},
}
for method_key, style in method_styles.items():
    mdata = thresh_data[(thresh_data["structure"] == "mitochondria") &
                        (thresh_data["method"] == method_key)].sort_values("threshold")
    if len(mdata) > 0 and mdata["dice"].max() > 0:
        ax.plot(mdata["threshold"], mdata["dice"],
                label=style["label"], color=style["color"],
                linestyle=style["ls"], marker=style["marker"], markersize=3, linewidth=1.5)
ax.set_ylabel("Dice Coefficient")
ax.set_title("Mitochondria", fontweight="bold", color=MITO_COLOR)
ax.set_xlim(0.05, 0.75)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=7, loc="lower left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.3)

# --- Membrane (top-right) ---
ax = axes[0, 1]
for method_key, style in method_styles.items():
    mdata = thresh_data[(thresh_data["structure"] == "membrane") &
                        (thresh_data["method"] == method_key)].sort_values("threshold")
    if len(mdata) > 0 and mdata["dice"].max() > 0:
        ax.plot(mdata["threshold"], mdata["dice"],
                label=style["label"], color=style["color"],
                linestyle=style["ls"], marker=style["marker"], markersize=3, linewidth=1.5)
ax.set_title("Membrane", fontweight="bold", color=MEMBRANE_COLOR)
ax.set_xlim(0.05, 0.75)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=7, loc="lower left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.3)

# --- Ribbon (bottom-left) ---
ax = axes[1, 0]
ribbon_thresh = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
                 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
ribbon_dice =   [0.001, 0.006, 0.082, 0.154, 0.206, 0.245, 0.276, 0.312, 0.354,
                 0.421, 0.505, 0.557, 0.590, 0.628, 0.672, 0.762, 0.743, 0.532]
ax.plot(ribbon_thresh, ribbon_dice, label="Fine-tuned (focal)", color="#9C27B0",
        linestyle="-", marker="o", markersize=3, linewidth=1.5)
best_idx = np.argmax(ribbon_dice)
ax.plot(ribbon_thresh[best_idx], ribbon_dice[best_idx], "r*", markersize=10, zorder=5)
ax.annotate(f"Best: {ribbon_dice[best_idx]:.3f}\n(t={ribbon_thresh[best_idx]})",
            xy=(ribbon_thresh[best_idx], ribbon_dice[best_idx]),
            xytext=(0.45, 0.8), fontsize=7,
            arrowprops=dict(arrowstyle="->", color="red", lw=0.8))
ax.set_xlabel("Threshold")
ax.set_ylabel("Dice Coefficient")
ax.set_title("Ribbon", fontweight="bold", color=RIBBON_COLOR)
ax.set_xlim(0.0, 0.95)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=7, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.3)

# --- Vesicles (bottom-right) ---
ax = axes[1, 1]
vesicle_thresh = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
                  0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
vesicle_dice =   [0.206, 0.144, 0.111, 0.090, 0.068, 0.052, 0.042, 0.024, 0.016,
                  0.012, 0.009, 0.006, 0.006, 0.003, 0.000]
ax.plot(vesicle_thresh, vesicle_dice, label="Retrained + ribbon prox (r=50)",
        color="#E65100", linestyle="-", marker="o", markersize=3, linewidth=1.5)
best_idx = np.argmax(vesicle_dice)
ax.plot(vesicle_thresh[best_idx], vesicle_dice[best_idx], "r*", markersize=10, zorder=5)
ax.annotate(f"Best: {vesicle_dice[best_idx]:.3f}\n(t={vesicle_thresh[best_idx]})",
            xy=(vesicle_thresh[best_idx], vesicle_dice[best_idx]),
            xytext=(0.35, 0.25), fontsize=7,
            arrowprops=dict(arrowstyle="->", color="red", lw=0.8))
ax.set_xlabel("Threshold")
ax.set_title("Vesicles", fontweight="bold", color=VESICLE_COLOR)
ax.set_xlim(0.0, 0.80)
ax.set_ylim(0, 0.35)
ax.legend(fontsize=7, loc="upper right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIG_DIR / "fig3_threshold_sweep.png", dpi=300)
fig.savefig(FIG_DIR / "fig3_threshold_sweep.tiff", dpi=300)
plt.close(fig)
print("Figure 3 saved: threshold sweep (all 4 structures)")

Figure 3 saved: threshold sweep (all 4 structures)


In [5]:
# Figure 4: Final optimized performance — all 4 structures
# Bar chart showing best Dice, IoU, Precision, Recall for each structure
# Each metric gets its own distinct color

structures = ["Mitochondria", "Membrane", "Ribbon", "Vesicles"]
struct_colors = [MITO_COLOR, MEMBRANE_COLOR, RIBBON_COLOR, VESICLE_COLOR]

best_dice = [0.869, 0.892, 0.762, 0.206]
best_iou = [0.769, 0.806, 0.615, 0.115]
best_precision = [0.800, 0.958, 0.798, 0.203]
best_recall = [0.952, 0.835, 0.729, 0.209]

# Distinct metric colors
DICE_COLOR = "#2196F3"       # blue
IOU_COLOR = "#FF9800"        # orange
PRECISION_COLOR = "#4CAF50"  # green
RECALL_COLOR = "#9C27B0"     # purple

x = np.arange(len(structures))
width = 0.20

fig, ax = plt.subplots(figsize=(7, 4.5))

bars1 = ax.bar(x - 1.5*width, best_dice, width, label="Dice",
               color=DICE_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)
bars2 = ax.bar(x - 0.5*width, best_iou, width, label="IoU",
               color=IOU_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)
bars3 = ax.bar(x + 0.5*width, best_precision, width, label="Precision",
               color=PRECISION_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)
bars4 = ax.bar(x + 1.5*width, best_recall, width, label="Recall",
               color=RECALL_COLOR, edgecolor="black", linewidth=0.5, alpha=0.85)

# Add Dice value labels
for i, bar in enumerate(bars1):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.015,
            f"{h:.3f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

# Method annotations below bars
method_labels = [
    "Ensemble (3 models)\nTTA + thresh opt",
    "Single model\nThresh opt only",
    "Single model\nThresh opt only",
    "Retrained model\nRibbon proximity (r=50)",
]
for i, label in enumerate(method_labels):
    ax.text(x[i], -0.08, label, ha="center", va="top", fontsize=6.5,
            fontstyle="italic", color="gray")

ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels(structures, fontsize=10, fontweight="bold")
ax.set_ylim(-0.15, 1.12)
ax.legend(loc="upper right", framealpha=0.9, ncol=4, fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axhline(y=0, color="black", linewidth=0.5)

plt.tight_layout()
fig.savefig(FIG_DIR / "fig4_optimization_progression.png", dpi=300)
fig.savefig(FIG_DIR / "fig4_optimization_progression.tiff", dpi=300)
plt.close(fig)
print("Figure 4 saved: final optimized performance (all 4 structures)")

Figure 4 saved: final optimized performance (all 4 structures)


In [ ]:
# Verify all figures exist
import os
for f in ["fig1_qualitative", "fig2_method_comparison", "fig3_threshold_sweep",
          "fig4_optimization_progression"]:
    png = FIG_DIR / f"{f}.png"
    tiff = FIG_DIR / f"{f}.tiff"
    png_sz = os.path.getsize(png)/1024 if png.exists() else 0
    tiff_sz = os.path.getsize(tiff)/1024 if tiff.exists() else 0
    print(f"{f}: PNG={png_sz:.0f}KB, TIFF={tiff_sz:.0f}KB")
print("\nFigures 1-4 generated successfully!")

## Figure 5: 3D Reconstruction of Ground Truth Annotations
Marching cubes mesh extraction from 3 serial TEM sections. Each section is assigned
a standardized thickness (20 voxels) and Gaussian smoothing along z creates smooth
transitions between sections. All panels use the same spatial extent for size comparison.

**Requires**: Node01 server (GT labels at `/mnt/Projects/SynapseNet-Retina/`)

Physical scale: XY = 5.0 nm/px, Z = ~54 nm section spacing (10.8:1 anisotropy)

In [ ]:
# Figure 5: 3D render of GT annotations using marching cubes
# Run this cell on node01 where GT labels are available
import numpy as np
import tifffile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter
import matplotlib.patches as mpatches

# GT label paths (node01)
GT_BASE = Path("/mnt/Projects/SynapseNet-Retina/output/run_006_finetune_baseline_binary_dice_20260227_202231/training_data")
RIBBON_GT_BASE = Path("/mnt/Projects/SynapseNet-Retina/output/run_ribbon_finetune_focal_20260301_061114/training_data")

STRUCTURES_3D = {
    "Mitochondria": {
        "path": GT_BASE / "mitochondria" / "labels",
        "color": (0.2, 0.8, 0.2, 0.7),
        "edge": (0.1, 0.5, 0.1, 0.3),
    },
    "Membrane": {
        "path": GT_BASE / "membrane" / "labels",
        "color": (0.2, 0.4, 1.0, 0.15),
        "edge": (0.1, 0.2, 0.6, 0.1),
    },
    "Ribbon": {
        "path": RIBBON_GT_BASE / "ribbon" / "labels",
        "color": (0.85, 0.15, 0.85, 0.8),
        "edge": (0.5, 0.0, 0.5, 0.3),
    },
    "Vesicles": {
        "path": GT_BASE / "vesicles" / "labels",
        "color": (1.0, 0.7, 0.0, 0.8),
        "edge": (0.7, 0.4, 0.0, 0.3),
    },
}

SECTION_THICKNESS = 20  # voxels per section in expanded volume
N_SECTIONS = 3
DOWNSAMPLE = 4

# Load GT labels
print("Loading GT labels...")
volumes = {}
for name, cfg in STRUCTURES_3D.items():
    slices = []
    for sec in ["section_000.tif", "section_001.tif", "section_002.tif"]:
        img = tifffile.imread(cfg["path"] / sec)
        slices.append((img > 0).astype(np.float32))
    vol = np.stack(slices, axis=0)
    volumes[name] = vol
    print(f"  {name}: {vol.shape}, foreground voxels: {vol.sum():.0f}")

H, W = volumes["Mitochondria"].shape[1], volumes["Mitochondria"].shape[2]

# Compute global bounding box
bboxes = {}
for name, vol in volumes.items():
    yz = np.where(vol.any(axis=0))
    if len(yz[0]) > 0:
        bboxes[name] = (int(yz[0].min()), int(yz[0].max()),
                        int(yz[1].min()), int(yz[1].max()))

all_y_min = min(b[0] for b in bboxes.values())
all_y_max = max(b[1] for b in bboxes.values())
all_x_min = min(b[2] for b in bboxes.values())
all_x_max = max(b[3] for b in bboxes.values())
global_bbox = (all_y_min, all_y_max, all_x_min, all_x_max)


def crop_volume(vol, bbox, pad_frac=0.20):
    y_min, y_max, x_min, x_max = bbox
    y_range = max(y_max - y_min, 200)
    x_range = max(x_max - x_min, 200)
    cy, cx = (y_min + y_max) // 2, (x_min + x_max) // 2
    half_y = y_range // 2 + int(y_range * pad_frac)
    half_x = x_range // 2 + int(x_range * pad_frac)
    y0 = max(0, cy - half_y)
    y1 = min(vol.shape[1], cy + half_y)
    x0 = max(0, cx - half_x)
    x1 = min(vol.shape[2], cx + half_x)
    return vol[:, y0:y1, x0:x1]


def add_structure_mesh(ax, vol, color, edge_color, downsample=DOWNSAMPLE,
                       section_thickness=SECTION_THICKNESS):
    ds = vol[:, ::downsample, ::downsample]
    n_sec, ds_h, ds_w = ds.shape
    total_z = n_sec * section_thickness
    expanded = np.zeros((total_z, ds_h, ds_w), dtype=np.float32)
    for i in range(n_sec):
        expanded[i * section_thickness:(i + 1) * section_thickness] = ds[i]
    smoothed = gaussian_filter(expanded, sigma=(3, 1, 1))
    padded = np.pad(smoothed, pad_width=1, mode='constant', constant_values=0)
    if (smoothed > 0.2).sum() < 5:
        print(f"    Skipping - too few voxels")
        return
    try:
        verts, faces, _, _ = marching_cubes(padded, level=0.3)
    except Exception as e:
        print(f"    Marching cubes failed: {e}")
        return
    verts[:, 0] = (verts[:, 0] - 1) / section_thickness
    verts[:, 1] = (verts[:, 1] - 1) * downsample
    verts[:, 2] = (verts[:, 2] - 1) * downsample
    mesh = Poly3DCollection(verts[faces], alpha=color[3])
    mesh.set_facecolor(color[:3])
    mesh.set_edgecolor(edge_color)
    mesh.set_linewidth(0.1)
    ax.add_collection3d(mesh)


def setup_ax(ax, title, crop_h, crop_w, elev=30, azim=-55):
    ax.set_xlim(0, N_SECTIONS)
    ax.set_ylim(0, crop_h)
    ax.set_zlim(0, crop_w)
    ax.set_xlabel("Section", fontsize=8, labelpad=2)
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=0)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(["0", "1", "2", "3"], fontsize=7)
    ax.set_yticks([])
    ax.set_zticks([])
    ax.view_init(elev=elev, azim=azim)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('lightgray')
    ax.yaxis.pane.set_edgecolor('lightgray')
    ax.zaxis.pane.set_edgecolor('lightgray')
    ax.grid(True, alpha=0.2)


# Generate figure
print("Generating 3D renders...")
fig = plt.figure(figsize=(14, 9))

panels = [
    (2, 3, 1, "Mitochondria"),
    (2, 3, 2, "Membrane"),
    (2, 3, 3, "Ribbon"),
    (2, 3, 4, "Vesicles"),
    (2, 3, 5, "All Structures"),
]

ref_crop = crop_volume(volumes["Mitochondria"], global_bbox)
crop_h, crop_w = ref_crop.shape[1], ref_crop.shape[2]
print(f"  Global crop: {crop_h} x {crop_w}")

for row, col, idx, title in panels:
    print(f"  Rendering {title}...")
    ax = fig.add_subplot(row, col, idx, projection='3d')
    if title == "All Structures":
        for name, cfg in STRUCTURES_3D.items():
            cropped = crop_volume(volumes[name], global_bbox)
            add_structure_mesh(ax, cropped, cfg["color"], cfg["edge"])
    else:
        cfg = STRUCTURES_3D[title]
        cropped = crop_volume(volumes[title], global_bbox)
        add_structure_mesh(ax, cropped, cfg["color"], cfg["edge"])
    setup_ax(ax, title, crop_h, crop_w)

# Legend panel
ax_legend = fig.add_subplot(2, 3, 6)
ax_legend.axis("off")
legend_items = []
for name, cfg in STRUCTURES_3D.items():
    patch = mpatches.Patch(color=cfg["color"][:3], alpha=0.8, label=name)
    legend_items.append(patch)
ax_legend.legend(handles=legend_items, loc="center", fontsize=12,
                 frameon=True, fancybox=True, shadow=True,
                 title="Structures", title_fontsize=13)
ax_legend.text(0.5, 0.15,
    "3D reconstruction from\n3 serial TEM sections\n(ground truth annotations)",
    ha="center", va="center", fontsize=9, fontstyle="italic",
    color="gray", transform=ax_legend.transAxes)

plt.tight_layout()
fig.savefig(FIG_DIR / "fig5_3d_render.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "fig5_3d_render.tiff", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Figure 5 saved: 3D reconstruction")

## Figure 6: Qualitative Full-Montage Evaluation
Compares fine-tuned 2D models (our pipeline) vs pretrained 3D SynapseNet models (zero-shot)
on full 7000×7000 TEM montages from 4 stacks (m0–m3). Demonstrates:
- **Part A**: Fine-tuned 2D models generalize across full montages (not just annotated crop)
- **Part B**: Pretrained 3D models (cryo-ET domain) produce blank/near-blank results → domain gap

**Requires**: Node01 GPU server (full montages + all model checkpoints)

### Model configurations:
| Structure | Method | Models | Threshold | TTA | Ensemble |
|-----------|--------|--------|-----------|-----|----------|
| Mitochondria | Focal FT | 3 (run008, DA, DA+FT) | 0.35 | 7 aug | Max-vote |
| Membrane | Focal FT | 1 (run008) | 0.40 | No | No |
| Ribbon | Focal FT | 1 (ribbon_best) | 0.80 | No | No |
| Vesicles | Retrained + ribbon prox | 1 (retrain_ribbon) | 0.05 | No | No |

### Pretrained 3D models tested:
mitochondria, compartments, ribbon (with vesicle context), vesicles_3d, active_zone

In [ ]:
# Figure 6: Qualitative evaluation on full 7000x7000 TEM montages
# Part A: Fine-tuned 2D models | Part B: Pretrained 3D SynapseNet (zero-shot)
# Run this cell on node01 where montages and checkpoints are available

import sys
import time
import numpy as np
import tifffile
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.morphology import opening, disk, label, remove_small_objects
from scipy.ndimage import distance_transform_edt

# --- Monkey-patch for checkpoint compatibility ---
import __main__
def _dummy_transform(*args, **kwargs):
    return None
__main__.binary_label_transform = _dummy_transform
__main__.label_transform = _dummy_transform

# Paths
STACK_DIR = Path("/mnt/Projects/SynapseNet-Retina/data/thiru-em-stacks")
CKPT_DIR = Path("/mnt/Projects/SynapseNet-Retina/models-env/finetuned/checkpoints/protected")
OUT_DIR = Path("/tmp/qualitative_eval")
OUT_DIR.mkdir(exist_ok=True)

STACKS = ["m0", "m1", "m2", "m3"]
TILE_SIZE = 256
TILE_STRIDE = 192
MORPH_RADIUS = 2
OVERLAY_ALPHA = 0.35

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Model configurations (matches webapp)
FINETUNED_MODELS = {
    "Mitochondria": {
        "checkpoints": [
            CKPT_DIR / "retina_mitochondria_best_run008.pt",
            CKPT_DIR / "retina_mitochondria_da.pt",
            CKPT_DIR / "retina_mitochondria_da_ft.pt",
        ],
        "threshold": 0.35, "min_area": 5000,
        "tta": True, "ensemble": True,
        "color": (0, 255, 0),
    },
    "Membrane": {
        "checkpoints": [CKPT_DIR / "retina_membrane_best_run008.pt"],
        "threshold": 0.40, "min_area": 50000,
        "tta": False, "ensemble": False,
        "color": (0, 0, 255),
    },
    "Ribbon": {
        "checkpoints": [CKPT_DIR / "retina_ribbon_best.pt"],
        "threshold": 0.80, "min_area": 100,
        "tta": False, "ensemble": False,
        "color": (255, 0, 255),
    },
    "Vesicles": {
        "checkpoints": [CKPT_DIR / "retina_vesicles_retrain_ribbon.pt"],
        "threshold": 0.05, "min_area": 20,
        "tta": False, "ensemble": False,
        "ribbon_radius": 50,
        "color": (0, 255, 255),
    },
}

PRETRAINED_3D_MODELS = {
    "Mitochondria (3D)": {"model_name": "mitochondria", "color": (0, 255, 0), "threshold": 0.5},
    "Compartments (3D)": {"model_name": "compartments", "color": (0, 0, 255), "threshold": 0.5},
    "Ribbon (3D)": {"model_name": "ribbon", "color": (255, 0, 255), "threshold": 0.5, "needs_vesicle_context": True},
    "Vesicles (3D)": {"model_name": "vesicles_3d", "color": (0, 255, 255), "threshold": 0.5},
    "Active Zone (3D)": {"model_name": "active_zone", "color": (255, 165, 0), "threshold": 0.5},
}

# TTA augmentations
TTA_TRANSFORMS = [
    ("identity",    lambda x: x,                        lambda x: x),
    ("hflip",       lambda x: np.flip(x, axis=1).copy(), lambda x: np.flip(x, axis=1).copy()),
    ("vflip",       lambda x: np.flip(x, axis=0).copy(), lambda x: np.flip(x, axis=0).copy()),
    ("hvflip",      lambda x: np.flip(np.flip(x, 0), 1).copy(), lambda x: np.flip(np.flip(x, 0), 1).copy()),
    ("rot90",       lambda x: np.rot90(x, 1).copy(),    lambda x: np.rot90(x, -1).copy()),
    ("rot180",      lambda x: np.rot90(x, 2).copy(),    lambda x: np.rot90(x, -2).copy()),
    ("rot270",      lambda x: np.rot90(x, 3).copy(),    lambda x: np.rot90(x, -3).copy()),
]

In [ ]:
# Helper functions for 2D and 3D inference

def load_2d_model(checkpoint_path):
    """Load a fine-tuned 2D UNet from checkpoint."""
    from torch_em.model import UNet2d
    save_dict = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
    if isinstance(save_dict, dict) and "model_state" in save_dict:
        state_dict = save_dict["model_state"]
    elif isinstance(save_dict, dict) and "state_dict" in save_dict:
        state_dict = save_dict["state_dict"]
    else:
        state_dict = save_dict
    out_channels = state_dict["out_conv.weight"].shape[0]
    has_sigmoid = out_channels == 2
    final_act = "Sigmoid" if has_sigmoid else None
    model = UNet2d(in_channels=1, out_channels=out_channels, depth=4,
                   initial_features=32, final_activation=final_act)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model, has_sigmoid


def tiled_inference_2d(model, image, has_sigmoid=False):
    """Run tiled inference on a 2D image. Returns probability map."""
    h, w = image.shape
    img_min, img_max = image.min(), image.max()
    if img_max > img_min:
        img_norm = (image.astype(np.float32) - img_min) / (img_max - img_min)
    else:
        img_norm = np.zeros_like(image, dtype=np.float32)
    prob_map = np.zeros((h, w), dtype=np.float64)
    count_map = np.zeros((h, w), dtype=np.float64)
    with torch.no_grad():
        for y in range(0, h, TILE_STRIDE):
            for x in range(0, w, TILE_STRIDE):
                ye = min(y + TILE_SIZE, h)
                xe = min(x + TILE_SIZE, w)
                patch = img_norm[y:ye, x:xe]
                padded = np.zeros((TILE_SIZE, TILE_SIZE), dtype=np.float32)
                padded[:ye-y, :xe-x] = patch
                tensor = torch.from_numpy(padded).unsqueeze(0).unsqueeze(0).to(device)
                output = model(tensor)
                if not has_sigmoid:
                    output = torch.sigmoid(output)
                pred = output[0, 0].cpu().numpy()
                prob_map[y:ye, x:xe] += pred[:ye-y, :xe-x]
                count_map[y:ye, x:xe] += 1.0
    count_map[count_map == 0] = 1.0
    return (prob_map / count_map).astype(np.float32)


def run_tta(model, image, has_sigmoid=False):
    """Run TTA with 7 geometric augmentations, return averaged prob map."""
    prob_maps = []
    for name, fwd, inv in TTA_TRANSFORMS:
        aug_img = fwd(image)
        prob = tiled_inference_2d(model, aug_img, has_sigmoid)
        prob_maps.append(inv(prob))
    return np.mean(prob_maps, axis=0).astype(np.float32)


def postprocess(binary, min_area):
    """Morphological opening + min area filtering."""
    opened = opening(binary, disk(MORPH_RADIUS))
    labeled = label(opened)
    cleaned = remove_small_objects(labeled, min_size=min_area)
    return (cleaned > 0).astype(np.uint8)


def run_finetuned_2d(image, structure_name, cfg, ribbon_mask=None):
    """Run full fine-tuned 2D pipeline for one structure."""
    t0 = time.time()
    prob_maps = []
    for ckpt_path in cfg["checkpoints"]:
        model, has_sigmoid = load_2d_model(ckpt_path)
        if cfg["tta"]:
            prob = run_tta(model, image, has_sigmoid)
        else:
            prob = tiled_inference_2d(model, image, has_sigmoid)
        prob_maps.append(prob)
        del model
        torch.cuda.empty_cache()
    if cfg["ensemble"] and len(prob_maps) > 1:
        prob_map = np.maximum.reduce(prob_maps)
    else:
        prob_map = prob_maps[0]
    if "ribbon_radius" in cfg and ribbon_mask is not None:
        dist = distance_transform_edt(ribbon_mask == 0)
        proximity_mask = (dist <= cfg["ribbon_radius"]).astype(np.float32)
        prob_map = prob_map * proximity_mask
    binary = (prob_map >= cfg["threshold"]).astype(np.uint8)
    mask = postprocess(binary, cfg["min_area"])
    dt = time.time() - t0
    print(f"    {structure_name}: {dt:.1f}s, fg={mask.sum()/(mask.size)*100:.2f}%")
    return mask, prob_map


def run_pretrained_3d(volume_3sec, model_name, threshold=0.5, extra_segmentation=None):
    """Run pretrained 3D SynapseNet model on 3-section volume.
    Mirror-pads z from 3 to 8 for 3D U-Net compatibility."""
    from synapse_net.inference import get_model, run_segmentation
    t0 = time.time()
    model = get_model(model_name, device=device)
    vol = volume_3sec.astype(np.float32)
    v_min, v_max = vol.min(), vol.max()
    if v_max > v_min:
        vol = (vol - v_min) / (v_max - v_min)
    padded = np.pad(vol, ((2, 3), (0, 0), (0, 0)), mode='reflect')
    mid_idx = 3
    tiling = {"tile": {"z": 8, "y": 512, "x": 512}, "halo": {"z": 0, "y": 64, "x": 64}}
    extra_kwargs = {}
    if model_name == "ribbon":
        if extra_segmentation is not None:
            padded_vesicle = np.pad(
                np.stack([extra_segmentation] * 3, axis=0),
                ((2, 3), (0, 0), (0, 0)), mode='reflect')
            extra_kwargs["extra_segmentation"] = padded_vesicle
        else:
            extra_kwargs["extra_segmentation"] = None
    try:
        result = run_segmentation(padded, model, model_name, tiling=tiling,
                                  verbose=True, **extra_kwargs)
        if isinstance(result, dict):
            seg = list(result.values())[0]
        else:
            seg = result
        mask = seg[mid_idx] if seg.ndim == 3 else seg
        if mask.dtype in [np.float32, np.float64]:
            mask = (mask >= threshold).astype(np.uint8)
        else:
            mask = (mask > 0).astype(np.uint8)
    except Exception as e:
        print(f"    {model_name} FAILED: {e}")
        import traceback
        traceback.print_exc()
        mask = np.zeros(volume_3sec.shape[1:], dtype=np.uint8)
    dt = time.time() - t0
    print(f"    {model_name}: {dt:.1f}s, fg={mask.sum()/mask.size*100:.2f}%")
    del model
    torch.cuda.empty_cache()
    return mask


def make_overlay(image, mask, color, alpha=OVERLAY_ALPHA):
    """Create RGB overlay of mask on grayscale image."""
    img = image.astype(np.float32)
    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min) * 255
    img = img.astype(np.uint8)
    rgb = np.stack([img, img, img], axis=-1).astype(np.float32)
    overlay_color = np.array(color, dtype=np.float32)
    mask_bool = mask > 0
    for c in range(3):
        rgb[:, :, c][mask_bool] = alpha * overlay_color[c] + (1 - alpha) * rgb[:, :, c][mask_bool]
    return rgb.astype(np.uint8)

print("Inference helpers loaded.")

In [ ]:
# Load all montage stacks
print("Loading full montage stacks...")
middle_sections = {}
full_stacks = {}
for stack in STACKS:
    sections = []
    for i in range(3):
        path = STACK_DIR / f"{stack}_{i:04d}.tif"
        img = tifffile.imread(str(path))
        sections.append(img)
        if i == 1:
            middle_sections[stack] = img
            print(f"  {stack}_0001: {img.shape}, dtype={img.dtype}, range=[{img.min()}, {img.max()}]")
    full_stacks[stack] = np.stack(sections, axis=0)

# --- PART A: Fine-tuned 2D Models ---
print("\n" + "=" * 70)
print("PART A: Fine-tuned 2D Models")
print("=" * 70)

ft_results = {}
for stack in STACKS:
    print(f"\n  Stack {stack}:")
    image = middle_sections[stack]
    ft_results[stack] = {}
    # Run ribbon first (needed for vesicle masking)
    ribbon_cfg = FINETUNED_MODELS["Ribbon"]
    ribbon_mask, _ = run_finetuned_2d(image, "Ribbon", ribbon_cfg)
    ft_results[stack]["Ribbon"] = ribbon_mask
    for struct_name, cfg in FINETUNED_MODELS.items():
        if struct_name == "Ribbon":
            continue
        if struct_name == "Vesicles":
            mask, _ = run_finetuned_2d(image, struct_name, cfg, ribbon_mask=ribbon_mask)
        else:
            mask, _ = run_finetuned_2d(image, struct_name, cfg)
        ft_results[stack][struct_name] = mask

# Create Figure A
print("\n  Creating Figure A (Fine-tuned 2D)...")
struct_order = ["Mitochondria", "Membrane", "Ribbon", "Vesicles"]
n_rows = 1 + len(struct_order)
n_cols = len(STACKS)
fig_a, axes_a = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
fig_a.suptitle("Fine-tuned 2D Models — Full Montage Predictions",
               fontsize=16, fontweight="bold", y=0.995)
for ci, stack in enumerate(STACKS):
    image = middle_sections[stack]
    axes_a[0, ci].imshow(image, cmap="gray")
    axes_a[0, ci].set_title(f"{stack} (middle section)", fontsize=10)
    axes_a[0, ci].axis("off")
    for ri, struct_name in enumerate(struct_order):
        mask = ft_results[stack][struct_name]
        color = FINETUNED_MODELS[struct_name]["color"]
        overlay = make_overlay(image, mask, color)
        axes_a[ri + 1, ci].imshow(overlay)
        if ci == 0:
            axes_a[ri + 1, ci].set_ylabel(struct_name, fontsize=11, fontweight="bold")
        axes_a[ri + 1, ci].axis("off")
plt.tight_layout()
fig_a.savefig(OUT_DIR / "fig_A_finetuned_2d.png", dpi=150, bbox_inches="tight")
fig_a.savefig(OUT_DIR / "fig_A_finetuned_2d.tiff", dpi=150, bbox_inches="tight")
plt.close(fig_a)
print(f"  Saved: {OUT_DIR / 'fig_A_finetuned_2d.png'}")

In [ ]:
# --- PART B: Pretrained 3D SynapseNet Models (zero-shot) ---
print("=" * 70)
print("PART B: Pretrained 3D SynapseNet Models (zero-shot)")
print("=" * 70)

pt3d_results = {}
for stack in STACKS:
    print(f"\n  Stack {stack}:")
    volume = full_stacks[stack]
    pt3d_results[stack] = {}
    # Run vesicles_3d first (needed as context for ribbon model)
    vesicle_3d_mask = None
    for display_name, cfg in PRETRAINED_3D_MODELS.items():
        if cfg["model_name"] == "vesicles_3d":
            mask = run_pretrained_3d(volume, cfg["model_name"], cfg["threshold"])
            pt3d_results[stack][display_name] = mask
            vesicle_3d_mask = mask
            break
    # Run remaining models
    for display_name, cfg in PRETRAINED_3D_MODELS.items():
        if cfg["model_name"] == "vesicles_3d":
            continue
        if cfg.get("needs_vesicle_context"):
            mask = run_pretrained_3d(volume, cfg["model_name"], cfg["threshold"],
                                     extra_segmentation=vesicle_3d_mask)
        else:
            mask = run_pretrained_3d(volume, cfg["model_name"], cfg["threshold"])
        pt3d_results[stack][display_name] = mask

# Create Figure B
print("\n  Creating Figure B (Pretrained 3D)...")
pt3d_order = list(PRETRAINED_3D_MODELS.keys())
n_rows_b = 1 + len(pt3d_order)
n_cols_b = len(STACKS)
fig_b, axes_b = plt.subplots(n_rows_b, n_cols_b, figsize=(n_cols_b * 5, n_rows_b * 5))
fig_b.suptitle("Pretrained 3D SynapseNet Models (Zero-Shot) — Full Montage",
               fontsize=16, fontweight="bold", y=0.995)
for ci, stack in enumerate(STACKS):
    image = middle_sections[stack]
    axes_b[0, ci].imshow(image, cmap="gray")
    axes_b[0, ci].set_title(f"{stack} (middle section)", fontsize=10)
    axes_b[0, ci].axis("off")
    for ri, display_name in enumerate(pt3d_order):
        mask = pt3d_results[stack][display_name]
        color = PRETRAINED_3D_MODELS[display_name]["color"]
        overlay = make_overlay(image, mask, color)
        axes_b[ri + 1, ci].imshow(overlay)
        if ci == 0:
            axes_b[ri + 1, ci].set_ylabel(display_name, fontsize=10, fontweight="bold")
        axes_b[ri + 1, ci].axis("off")
plt.tight_layout()
fig_b.savefig(OUT_DIR / "fig_B_pretrained_3d.png", dpi=150, bbox_inches="tight")
fig_b.savefig(OUT_DIR / "fig_B_pretrained_3d.tiff", dpi=150, bbox_inches="tight")
plt.close(fig_b)
print(f"  Saved: {OUT_DIR / 'fig_B_pretrained_3d.png'}")

# Save individual overlay TIFFs
print("\n  Saving individual overlay TIFFs...")
for stack in STACKS:
    image = middle_sections[stack]
    for struct_name in struct_order:
        mask = ft_results[stack][struct_name]
        color = FINETUNED_MODELS[struct_name]["color"]
        overlay = make_overlay(image, mask, color)
        fname = f"{stack}_finetuned_{struct_name.lower().replace(' ', '_')}.tiff"
        tifffile.imwrite(str(OUT_DIR / fname), overlay)
    for display_name in pt3d_order:
        mask = pt3d_results[stack][display_name]
        color = PRETRAINED_3D_MODELS[display_name]["color"]
        overlay = make_overlay(image, mask, color)
        clean_name = display_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
        fname = f"{stack}_pretrained_{clean_name}.tiff"
        tifffile.imwrite(str(OUT_DIR / fname), overlay)

print("\n" + "=" * 70)
print("DONE. Figure 6 outputs saved to:", OUT_DIR)
print("=" * 70)

## Part C: Post-hoc 3D Consistency Enforcement (D11)
Evaluates whether applying volumetric consistency to stacked 2D predictions improves
segmentation quality. Tests three methods as independent toggles:

1. **3D Connected Components**: Stack 2D masks → 3D CC → filter by volume (removes single-slice noise)
2. **Z-Smoothing**: Gaussian filter on probability maps along z → re-threshold (smooths predictions)
3. **Instance Tracking**: Link instances across adjacent slices by overlap (merge fragments, remove orphans)

With only 3 serial sections, the effect is limited, but this establishes the framework
for the extended dataset (≥8 sections). Each 2D configuration is evaluated both with
and without post-hoc 3D consistency.

**Requires**: Node01 GPU server (same setup as Part A — saves probability maps for z-smoothing)
**Reference**: See `DESIGN_DECISIONS.txt` D11, `PLAN_3D_PIPELINE.txt` Post-hoc section

In [ ]:
# Post-hoc 3D Consistency: Helper functions
# Applied to stacked 2D predictions to test volumetric coherence

from scipy.ndimage import gaussian_filter, label as ndi_label
from skimage.measure import regionprops

def posthoc_3d_connected_components(stacked_masks, min_volume_voxels=50):
    """Stack 2D binary masks → 3D connected components → filter by volume.
    
    Removes instances that appear on only one slice (likely noise) by
    requiring a minimum 3D volume. With 3 sections, a true structure
    should span ≥2 sections.
    
    Args:
        stacked_masks: (Z, H, W) binary uint8 array
        min_volume_voxels: minimum 3D connected component volume to retain
    
    Returns:
        filtered: (Z, H, W) binary uint8 array
    """
    labeled_3d, n_components = ndi_label(stacked_masks)
    if n_components == 0:
        return stacked_masks.copy()
    
    filtered = np.zeros_like(stacked_masks)
    for region in regionprops(labeled_3d):
        if region.area >= min_volume_voxels:
            filtered[labeled_3d == region.label] = 1
    
    return filtered.astype(np.uint8)


def posthoc_z_smoothing(stacked_probs, threshold, sigma_z=0.8, sigma_xy=0):
    """Gaussian filter on probability maps along z, then re-threshold.
    
    With 3 sections, sigma_z=0.8 provides mild smoothing without excessive
    blurring. The smoothed probabilities are re-thresholded at the same
    value used for the original 2D predictions.
    
    Args:
        stacked_probs: (Z, H, W) float32 probability maps
        threshold: binarization threshold
        sigma_z: Gaussian sigma along z-axis
        sigma_xy: Gaussian sigma along xy (0 = no xy smoothing)
    
    Returns:
        smoothed_masks: (Z, H, W) binary uint8 array
    """
    smoothed = gaussian_filter(stacked_probs, sigma=(sigma_z, sigma_xy, sigma_xy))
    return (smoothed >= threshold).astype(np.uint8)


def posthoc_instance_tracking(stacked_masks, min_overlap_fraction=0.1, min_slices=2):
    """Link instances across adjacent slices by overlap, remove orphans.
    
    For each instance in slice N, checks if it overlaps with any instance
    in slice N+1. Instances that appear on fewer than min_slices are removed
    as likely false positives.
    
    Args:
        stacked_masks: (Z, H, W) binary uint8 array
        min_overlap_fraction: minimum overlap (IoU) to link instances
        min_slices: minimum number of slices an instance must span
    
    Returns:
        tracked_masks: (Z, H, W) binary uint8 array
    """
    z_depth = stacked_masks.shape[0]
    
    # Label each slice independently
    per_slice_labels = []
    for z in range(z_depth):
        labeled, _ = ndi_label(stacked_masks[z])
        per_slice_labels.append(labeled)
    
    # Track instances across slices
    # Build a graph: (slice_idx, label_id) -> set of (slice_idx, label_id)
    instance_groups = []  # list of sets, each set = {(z, label)} for one tracked instance
    visited = set()
    
    def find_linked(z, lab):
        """BFS to find all linked instances across slices."""
        group = set()
        queue = [(z, lab)]
        while queue:
            cz, cl = queue.pop(0)
            if (cz, cl) in group:
                continue
            group.add((cz, cl))
            # Check adjacent slices
            for nz in [cz - 1, cz + 1]:
                if nz < 0 or nz >= z_depth:
                    continue
                mask_current = (per_slice_labels[cz] == cl)
                neighbor_labels = per_slice_labels[nz]
                overlapping = np.unique(neighbor_labels[mask_current])
                for nl in overlapping:
                    if nl == 0 or (nz, nl) in group:
                        continue
                    # Check overlap fraction
                    mask_neighbor = (neighbor_labels == nl)
                    intersection = np.logical_and(mask_current, mask_neighbor).sum()
                    union = np.logical_or(mask_current, mask_neighbor).sum()
                    if union > 0 and intersection / union >= min_overlap_fraction:
                        queue.append((nz, nl))
        return group
    
    # Find all connected groups
    for z in range(z_depth):
        labels_in_slice = np.unique(per_slice_labels[z])
        for lab in labels_in_slice:
            if lab == 0 or (z, lab) in visited:
                continue
            group = find_linked(z, lab)
            visited.update(group)
            instance_groups.append(group)
    
    # Filter: keep only groups spanning >= min_slices
    tracked = np.zeros_like(stacked_masks)
    for group in instance_groups:
        slices_spanned = len(set(z for z, _ in group))
        if slices_spanned >= min_slices:
            for z, lab in group:
                tracked[z][per_slice_labels[z] == lab] = 1
    
    return tracked.astype(np.uint8)


def compute_dice(pred, gt):
    """Compute Dice coefficient between binary masks."""
    pred_bool = pred.astype(bool)
    gt_bool = gt.astype(bool)
    intersection = np.logical_and(pred_bool, gt_bool).sum()
    total = pred_bool.sum() + gt_bool.sum()
    if total == 0:
        return 1.0 if pred_bool.sum() == 0 and gt_bool.sum() == 0 else 0.0
    return 2.0 * intersection / total


print("Post-hoc 3D consistency helpers loaded.")

In [ ]:
# Part C: Run post-hoc 3D consistency evaluation on annotated crop region
# Uses the annotated crop region (section 000–002) where we have GT labels
# Runs 2D inference on all 3 sections, stacks predictions, applies 3D post-hoc methods

import pandas as pd

# GT paths (same as training data — annotated crop region)
GT_BASE = Path("/mnt/Projects/SynapseNet-Retina/output/run_006_finetune_baseline_binary_dice_20260227_202231/training_data")
RIBBON_GT = Path("/mnt/Projects/SynapseNet-Retina/output/run_ribbon_finetune_focal_20260301_061114/training_data")
IMAGE_BASE = GT_BASE / "mitochondria" / "images"  # all structures share same images

STRUCTURE_GT_PATHS = {
    "Mitochondria": GT_BASE / "mitochondria" / "labels",
    "Membrane": GT_BASE / "membrane" / "labels",
    "Ribbon": RIBBON_GT / "ribbon" / "labels",
    "Vesicles": GT_BASE / "vesicles" / "labels",
}

SECTIONS = ["section_000.tif", "section_001.tif", "section_002.tif"]

# Load images and GT labels
print("Loading annotated crop images and GT labels...")
images_3sec = []
gt_stacks = {s: [] for s in STRUCTURE_GT_PATHS}

for sec in SECTIONS:
    img = tifffile.imread(str(IMAGE_BASE / sec))
    images_3sec.append(img)
    for struct_name, gt_path in STRUCTURE_GT_PATHS.items():
        gt = tifffile.imread(str(gt_path / sec))
        gt_stacks[struct_name].append((gt > 0).astype(np.uint8))
    print(f"  {sec}: image {img.shape}, dtype={img.dtype}")

images_3sec = np.stack(images_3sec, axis=0)  # (3, H, W)
gt_volumes = {s: np.stack(v, axis=0) for s, v in gt_stacks.items()}  # (3, H, W) each

for s, v in gt_volumes.items():
    print(f"  GT {s}: shape={v.shape}, fg_voxels={v.sum()}")

# Run 2D inference on ALL 3 sections (not just middle)
print("\n" + "=" * 70)
print("Running 2D inference on all 3 annotated crop sections...")
print("=" * 70)

pred_probs = {}   # {structure: (3, H, W) float32 probability maps}
pred_masks = {}   # {structure: (3, H, W) uint8 binary masks}

for struct_name, cfg in FINETUNED_MODELS.items():
    print(f"\n  {struct_name}:")
    probs_list = []
    masks_list = []
    
    # Get ribbon masks first (needed for vesicle proximity masking)
    ribbon_masks_3sec = None
    if struct_name == "Vesicles":
        if "Ribbon" in pred_masks:
            ribbon_masks_3sec = pred_masks["Ribbon"]
        else:
            print("    WARNING: Ribbon not yet processed, running ribbon first...")
            # Should not happen if we process in order: Ribbon before Vesicles
    
    for z_idx in range(3):
        image = images_3sec[z_idx]
        prob_maps_per_model = []
        
        for ckpt_path in cfg["checkpoints"]:
            model, has_sigmoid = load_2d_model(ckpt_path)
            if cfg["tta"]:
                prob = run_tta(model, image, has_sigmoid)
            else:
                prob = tiled_inference_2d(model, image, has_sigmoid)
            prob_maps_per_model.append(prob)
            del model
            torch.cuda.empty_cache()
        
        if cfg["ensemble"] and len(prob_maps_per_model) > 1:
            prob_map = np.maximum.reduce(prob_maps_per_model)
        else:
            prob_map = prob_maps_per_model[0]
        
        # Apply ribbon proximity for vesicles
        if "ribbon_radius" in cfg and ribbon_masks_3sec is not None:
            dist = distance_transform_edt(ribbon_masks_3sec[z_idx] == 0)
            proximity_mask = (dist <= cfg["ribbon_radius"]).astype(np.float32)
            prob_map = prob_map * proximity_mask
        
        binary = (prob_map >= cfg["threshold"]).astype(np.uint8)
        mask = postprocess(binary, cfg["min_area"])
        
        probs_list.append(prob_map)
        masks_list.append(mask)
        
        gt_slice = gt_volumes[struct_name][z_idx]
        dice_slice = compute_dice(mask, gt_slice)
        print(f"    Section {z_idx}: fg={mask.sum()/mask.size*100:.2f}%, Dice={dice_slice:.4f}")
    
    pred_probs[struct_name] = np.stack(probs_list, axis=0)
    pred_masks[struct_name] = np.stack(masks_list, axis=0)

# Apply post-hoc 3D consistency methods
print("\n" + "=" * 70)
print("Applying post-hoc 3D consistency methods...")
print("=" * 70)

# Min volume thresholds for 3D CC (in voxels, accounting for 3 slices)
MIN_VOL_3D = {
    "Mitochondria": 5000,   # large structures, span all 3 slices
    "Membrane": 10000,      # continuous, large
    "Ribbon": 100,          # small but should span ≥2 slices
    "Vesicles": 20,         # very small
}

results_rows = []

for struct_name in FINETUNED_MODELS:
    print(f"\n  {struct_name}:")
    stacked_masks = pred_masks[struct_name]    # (3, H, W) uint8
    stacked_probs = pred_probs[struct_name]    # (3, H, W) float32
    gt_vol = gt_volumes[struct_name]           # (3, H, W) uint8
    threshold = FINETUNED_MODELS[struct_name]["threshold"]
    
    # Baseline: original 2D predictions (no post-hoc)
    for z_idx in range(3):
        dice_2d = compute_dice(stacked_masks[z_idx], gt_vol[z_idx])
        results_rows.append({
            "structure": struct_name, "section": z_idx,
            "method": "2D (baseline)", "dice": dice_2d,
            "fg_pct": stacked_masks[z_idx].sum() / stacked_masks[z_idx].size * 100,
        })
    # Volumetric Dice (all 3 slices)
    dice_2d_vol = compute_dice(stacked_masks, gt_vol)
    print(f"    Baseline 2D:  volumetric Dice = {dice_2d_vol:.4f}")
    
    # Method 1: 3D Connected Components
    cc3d = posthoc_3d_connected_components(stacked_masks, min_volume_voxels=MIN_VOL_3D[struct_name])
    for z_idx in range(3):
        dice_cc = compute_dice(cc3d[z_idx], gt_vol[z_idx])
        results_rows.append({
            "structure": struct_name, "section": z_idx,
            "method": "3D-CC", "dice": dice_cc,
            "fg_pct": cc3d[z_idx].sum() / cc3d[z_idx].size * 100,
        })
    dice_cc_vol = compute_dice(cc3d, gt_vol)
    removed_pct = (stacked_masks.sum() - cc3d.sum()) / max(stacked_masks.sum(), 1) * 100
    print(f"    3D-CC:        volumetric Dice = {dice_cc_vol:.4f}  (removed {removed_pct:.1f}% of fg)")
    
    # Method 2: Z-Smoothing
    z_smooth = posthoc_z_smoothing(stacked_probs, threshold=threshold, sigma_z=0.8)
    z_smooth_pp = np.stack([
        postprocess(z_smooth[z], FINETUNED_MODELS[struct_name]["min_area"])
        for z in range(3)
    ], axis=0)
    for z_idx in range(3):
        dice_zs = compute_dice(z_smooth_pp[z_idx], gt_vol[z_idx])
        results_rows.append({
            "structure": struct_name, "section": z_idx,
            "method": "Z-smooth", "dice": dice_zs,
            "fg_pct": z_smooth_pp[z_idx].sum() / z_smooth_pp[z_idx].size * 100,
        })
    dice_zs_vol = compute_dice(z_smooth_pp, gt_vol)
    print(f"    Z-smoothing:  volumetric Dice = {dice_zs_vol:.4f}")
    
    # Method 3: Instance Tracking (min_slices=2 removes single-slice noise)
    tracked = posthoc_instance_tracking(stacked_masks, min_overlap_fraction=0.1, min_slices=2)
    for z_idx in range(3):
        dice_tr = compute_dice(tracked[z_idx], gt_vol[z_idx])
        results_rows.append({
            "structure": struct_name, "section": z_idx,
            "method": "Instance-track", "dice": dice_tr,
            "fg_pct": tracked[z_idx].sum() / tracked[z_idx].size * 100,
        })
    dice_tr_vol = compute_dice(tracked, gt_vol)
    removed_tr = (stacked_masks.sum() - tracked.sum()) / max(stacked_masks.sum(), 1) * 100
    print(f"    Inst-track:   volumetric Dice = {dice_tr_vol:.4f}  (removed {removed_tr:.1f}% of fg)")

# Save results
results_df = pd.DataFrame(results_rows)
results_df.to_csv(OUT_DIR / "posthoc_3d_consistency_results.csv", index=False)
print(f"\nResults saved to: {OUT_DIR / 'posthoc_3d_consistency_results.csv'}")

# Summary table
print("\n" + "=" * 70)
print("SUMMARY: Volumetric Dice (all 3 sections)")
print("=" * 70)
summary = results_df.groupby(["structure", "method"])["dice"].mean().unstack(fill_value=0)
print(summary.to_string(float_format="%.4f"))
print("\nNote: With only 3 sections, z-smoothing and instance tracking have limited effect.")
print("Framework ready for extended dataset (≥8 sections) where 3D consistency should matter more.")

## Summary
All figures and evaluations:
| Figure/Eval | Description | Compute |
|-------------|-------------|---------|
| Fig 1 | Qualitative comparison: EM / GT / Prediction for 4 structures | Local |
| Fig 2 | Method comparison bar chart (6 methods + extended structures) | Local |
| Fig 3 | Threshold sweep curves (4 structures) | Local |
| Fig 4 | Final optimized performance (Dice, IoU, Precision, Recall) | Local |
| Fig 5 | 3D reconstruction of GT annotations (marching cubes meshes) | Node01 |
| Fig 6A | Full-montage qualitative eval: fine-tuned 2D models | Node01 |
| Fig 6B | Full-montage qualitative eval: pretrained 3D zero-shot | Node01 |
| Part C | Post-hoc 3D consistency evaluation (D11) | Node01 |

### Design Decisions Documented
See `DESIGN_DECISIONS.txt` for all method evaluation decisions (D1–D14).

### Post-hoc 3D Consistency Methods (D11)
- **3D Connected Components**: Stack masks, filter by 3D volume → removes single-slice noise
- **Z-Smoothing**: Gaussian filter on probability maps along z → re-threshold
- **Instance Tracking**: Link instances across slices by overlap → remove orphan detections
- Each 2D configuration evaluated both with and without each post-hoc method
- Framework scales to extended dataset (≥8 sections) for meaningful 3D comparison

### Future Work
See `PLAN_3D_PIPELINE.txt` for extended 3D benchmarking plan:
- Architecture B: 2.5D U-Net (3-channel adjacent slices)
- Architecture C: 3D AnisotropicUNet (native volumetric)
- Full results table: architecture × method × structure × 3D-CC toggle